***Title: Classification Models Assignment***

***Name: Gedion Sang***

***ID: CS-DA03-26010***

***Date: 23/03/2026***

## Natural Language Processing
### Introduction
NLP is a subfield of linguistics and AI and whose primary focus is bulding modelsto demonstrate human-level understanding of written and spoken text.

Made up of two subfields;

    - Natural Language Understanding(NLU) - focuses on semantic analysis or determining intended meaning of text.
    
    - Natural Language Generation(NLG) - focus on generating meaningful text.
    
Key NLP tasks are tokenization, stemming, lemmatization, POS tagging, NER, parsing

### 1. Import Libraries 
`Perhaps the easiest step of the lot😁`

- importing the necessary libraries
- `BertTokenizer` and `BertModel` from the `transformers` library for working with BERT models.
- `numpy` for numerical operations
- `cosine_similarity` from `sklearn.metrics.pairwise` to calculate the similarity between sentence embeddings.


In [1]:

## Step 1: Import required libraries
from transformers import BertTokenizer, BertModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

### 2. Load BertTokenizer and Bert Model
This cell loads a pre-trained BERT tokenizer and model. It uses the `bert-base-uncased` version, which is a widely used general-purpose BERT model trained on uncased English text. The `bert_model.eval()` line sets the model to evaluation mode, which disables dropout and batch normalization, ensuring consistent predictions.


In [2]:
# Step 2: Load pre-trained BERT tokenizer and BERT model from Hugging Face
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()  # Set the model to evaluation mode  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

This cell defines a list of `sentence_pairs`. These pairs will be used to test the similarity detection capabilities of the BERT model. Some pairs are semantically similar, while others are distinct.

In [3]:
# Example sentence pairs (some similar, some not)
sentence_pairs = [
    ("How do I learn Python?", "What is the best way to study Python?"),
    ("What is AI?", "How to cook pasta?"),
    ("How do I bake a chocolate cake?", "Give me a chocolate cake recipe."),
    ("How can I improve my coding skills?", "Tips for becoming better at programming."),
    ("Where can I buy cheap laptops?", "Best sites to find affordable computers."),]

This cell defines the `ground truth` labels for the `sentence_pairs`. A `1` indicates that the corresponding sentence pair is considered similar, and `0` indicates it is not similar. This will be used to evaluate the model's performance.

In [4]:
# Ground truth similarity labels: 1 = similar, 0 = not similar
labels = [1, 0, 1, 1, 1]

This code block defines the `get_sentence_embedding` function. It takes a sentence as input, tokenizes it using the BERT tokenizer, and then feeds it to the BERT model to get the contextual embeddings. Specifically, it extracts the embedding of the `[CLS]` token, which is often used as a fixed-dimensional representation of the entire sentence. `torch.no_grad()` is used to disable gradient calculation, which is standard practice during inference to save memory and computation.

In [5]:
import torch
# Function to get the BERT [CLS] embedding for a sentence
def get_sentence_embedding(sentence):
    # Tokenize and encode sentence into input tensors
    inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)
    # Get model output
    with torch.no_grad():  # Disable gradient calculation for inference
        outputs = bert_model(**inputs)
    # Extract [CLS] token embedding (shape: [1, 768])
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    return cls_embedding.numpy()

This cell iterates through each `sentence_pair`, generates their BERT embeddings using the `get_sentence_embedding` function, and then calculates the cosine similarity between them. A `prediction` is made based on whether the `cosine similarity` score is above a threshold of `0.7`. The sentences, their similarity score, and the prediction are then printed.

In [6]:
# Calculate cosine similarity for each pair
predictions = []
for sent1, sent2 in sentence_pairs:
    emb1 = get_sentence_embedding(sent1)
    emb2 = get_sentence_embedding(sent2)
    sim_score = cosine_similarity(emb1, emb2)[0][0]
    pred = 1 if sim_score > 0.7 else 0
    predictions.append(pred)
    
    print(f"\nSentence 1: {sent1}")
    print(f"Sentence 2: {sent2}")
    print(f"Cosine Similarity: {sim_score:.4f} → Predicted Similar: {pred}")


Sentence 1: How do I learn Python?
Sentence 2: What is the best way to study Python?
Cosine Similarity: 0.9743 → Predicted Similar: 1

Sentence 1: What is AI?
Sentence 2: How to cook pasta?
Cosine Similarity: 0.9033 → Predicted Similar: 1

Sentence 1: How do I bake a chocolate cake?
Sentence 2: Give me a chocolate cake recipe.
Cosine Similarity: 0.8938 → Predicted Similar: 1

Sentence 1: How can I improve my coding skills?
Sentence 2: Tips for becoming better at programming.
Cosine Similarity: 0.8633 → Predicted Similar: 1

Sentence 1: Where can I buy cheap laptops?
Sentence 2: Best sites to find affordable computers.
Cosine Similarity: 0.8750 → Predicted Similar: 1


This code block evaluates the accuracy of the similarity predictions. It compares each `prediction` made by the model against its corresponding `ground truth label` and counts how many were correct.

In [7]:
# Evaluate accuracy 
correct = 0
for i in range(len(predictions)):
    if predictions[i] == labels[i]:
        correct += 1


This cell calculates and prints the final accuracy of the model's `sentence similarity predictions` by dividing the number of correct predictions by the total number of labels.

In [8]:

# Final accuracy calculation
total = len(labels)
accuracy = correct / total
print(f"\nAccuracy: {accuracy:.2%}")


Accuracy: 80.00%
